# NB 00: Data Collection

Acquire historical funding rates from all 7 exchanges (4 CEX + 3 DEX),
normalize to 8h resolution, align temporally, and generate a data quality report.

**Exchanges:** Binance, Bybit, OKX, BitMEX, dYdX, Drift, Hyperliquid  
**Assets:** BTC, ETH, SOL, XRP, BNB  
**Target:** 12 months of history (minimum 6 months)

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
from datetime import datetime, timezone, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import EXCHANGES, ASSETS, RAW_DIR, PROCESSED_DIR
from src.collectors.ccxt_collector import (
    collect_all_funding_rates as collect_cex_funding,
    collect_all_prices as collect_cex_prices,
)
from src.collectors.hyperliquid_collector import (
    collect_all_funding_rates as collect_hl_funding,
)
from src.collectors.drift_collector import (
    collect_all_funding_rates as collect_drift_funding,
)
from src.collectors.dydx_collector import (
    collect_all_funding_rates as collect_dydx_funding,
)
from src.processing.normalizer import normalize_all
from src.processing.aligner import align_and_save, compute_coverage
from src.processing.cleaner import clean_funding_rates, generate_quality_report

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

# Target start date: 12 months ago
SINCE = datetime.now(timezone.utc) - timedelta(days=365)
print(f'Collecting data since: {SINCE.date()}')
print(f'Exchanges: {list(EXCHANGES.keys())}')
print(f'Assets: {ASSETS}')

## 1. CEX Funding Rates (via CCXT)

Binance, Bybit, OKX, BitMEX — all have 8h funding intervals.
CCXT's `fetch_funding_rate_history` handles pagination.

In [ ]:
cex_funding = collect_cex_funding(since=SINCE)
print(f'\nCEX funding rates collected: {len(cex_funding)} pairs')
for key, df in sorted(cex_funding.items()):
    print(f'  {key}: {len(df)} records, {df["timestamp"].min().date()} to {df["timestamp"].max().date()}')

## 2. DEX Funding Rates

dYdX, Drift, Hyperliquid — all have 1h funding intervals.
Each uses a custom REST API.

In [ ]:
# dYdX v4
dydx_funding = collect_dydx_funding(since=SINCE)
print(f'\ndYdX funding rates collected: {len(dydx_funding)} pairs')
for key, df in sorted(dydx_funding.items()):
    print(f'  {key}: {len(df)} records, {df["timestamp"].min().date()} to {df["timestamp"].max().date()}')

In [ ]:
# Drift
drift_funding = collect_drift_funding(since=SINCE)
print(f'\nDrift funding rates collected: {len(drift_funding)} pairs')
for key, df in sorted(drift_funding.items()):
    print(f'  {key}: {len(df)} records, {df["timestamp"].min().date()} to {df["timestamp"].max().date()}')

In [ ]:
# Hyperliquid
hl_funding = collect_hl_funding(since=SINCE)
print(f'\nHyperliquid funding rates collected: {len(hl_funding)} pairs')
for key, df in sorted(hl_funding.items()):
    print(f'  {key}: {len(df)} records, {df["timestamp"].min().date()} to {df["timestamp"].max().date()}')

## 3. CEX Price Data (OHLCV)

Collect 1h OHLCV from one reference CEX (Binance) for mark price comparison
and basis risk analysis.

In [ ]:
cex_prices = collect_cex_prices(
    exchanges=['binance'],
    since=SINCE,
    timeframe='1h',
)
print(f'\nPrice data collected: {len(cex_prices)} pairs')
for key, df in sorted(cex_prices.items()):
    print(f'  {key}: {len(df)} candles, {df["timestamp"].min().date()} to {df["timestamp"].max().date()}')

## 4. Normalize to 8h Resolution

CEX rates are already 8h. DEX 1h rates are summed over 8h windows
aligned to standard settlement times (00:00, 08:00, 16:00 UTC).

In [ ]:
normalized = normalize_all()
print(f'\nNormalized: {len(normalized)} total 8h observations')
print(f'Exchanges: {normalized["exchange"].unique().tolist()}')
print(f'Assets: {normalized["asset"].unique().tolist()}')
print(f'\nPer exchange-asset counts:')
print(normalized.groupby(['exchange', 'asset']).size().unstack(fill_value=0))

## 5. Temporal Alignment

Pivot to wide format (one column per exchange-asset) aligned by timestamp.
BitMEX's 4h offset is shifted to align with standard windows.

In [ ]:
wide, coverage = align_and_save(normalized)
print(f'\nAligned shape: {wide.shape}')
print(f'Date range: {wide.index.min()} to {wide.index.max()}')
print(f'\nCoverage report:')
coverage

## 6. Data Quality & Cleaning

In [ ]:
cleaned = clean_funding_rates(normalized)
quality = generate_quality_report(cleaned)
print('Data quality report:')
quality

## 7. Coverage Visualization

In [ ]:
fig, axes = plt.subplots(len(ASSETS), 1, figsize=(14, 3 * len(ASSETS)), sharex=True)

for i, asset in enumerate(ASSETS):
    ax = axes[i]
    asset_data = normalized[normalized['asset'] == asset]
    
    for j, exchange in enumerate(sorted(asset_data['exchange'].unique())):
        ex_data = asset_data[asset_data['exchange'] == exchange]
        ax.scatter(
            ex_data['timestamp'], 
            [j] * len(ex_data),
            s=1, alpha=0.5, label=exchange
        )
    
    ax.set_yticks(range(len(asset_data['exchange'].unique())))
    ax.set_yticklabels(sorted(asset_data['exchange'].unique()))
    ax.set_title(f'{asset} Data Coverage')

plt.tight_layout()
plt.savefig('../figures/data_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Coverage plot saved to figures/data_coverage.png')

## 8. Summary Statistics

In [ ]:
# Quick sanity check: average funding rates should be positive (~0.01%/8h)
summary = normalized.groupby(['exchange', 'asset'])['funding_rate_8h'].agg(
    ['mean', 'median', 'std', 'min', 'max', 'count']
).round(6)

print('Average funding rates by exchange-asset (sanity check: should be ~0.0001 / 0.01% per 8h):')
print()
summary

In [ ]:
# Total data inventory
total_obs = len(normalized)
n_exchanges = normalized['exchange'].nunique()
n_assets = normalized['asset'].nunique()
n_pairs = normalized.groupby(['exchange', 'asset']).ngroups
date_range = f"{normalized['timestamp'].min().date()} to {normalized['timestamp'].max().date()}"

print(f'=== Data Collection Complete ===')
print(f'Total 8h observations: {total_obs:,}')
print(f'Exchanges: {n_exchanges}')
print(f'Assets: {n_assets}')
print(f'Exchange-asset pairs: {n_pairs}')
print(f'Date range: {date_range}')
print(f'\nFiles saved to: {PROCESSED_DIR}')